In [74]:
import pyvisa                               # para comunicarnos con los equipos
import numpy as np                          # para operar con listas de datos
import pandas as pd                         # para manejar archivos de datos: Guardar, abrir, etc
import matplotlib.pyplot as plt             # para plotear 
import time                                 # para realizar pausas entre mediciones

rm = pyvisa.ResourceManager()

instrumentos_conectados = rm.list_resources()           # lista de instrumentos conectados

print (instrumentos_conectados)

gen = rm.open_resource('USB0::0xF4EC::0xEE38::SDG1XCA4161035::INSTR')                     # Establezco comunicación con el generador

gen.query('*IDN?')                                      # le pedimos que se identifique para checkear conexión

osci = rm.open_resource('USB0::0x0699::0x03C4::C041039::INSTR')                    # Establezco comunicación con el osciloscopio

osci.query('*IDN?')           
# le pido que se identifique para checkear la conexión


('USB0::0xF4EC::0xEE38::SDG1XCA4161035::INSTR', 'USB0::0x0699::0x03C4::C041039::INSTR', 'ASRL1::INSTR', 'ASRL10::INSTR')


'TEKTRONIX,TBS1102C,C041039,CF:91.1CT FV:v1.29.38; FPGA:v20.78; \n'

In [75]:
gen.write('C1:OUTP OFF')

ampli= 4.            # voltaje pico a pico
freq = 1000.         # en Hz 
fase = 180.          # en grados

gen.write('C1:BSWV WVTP,SINE,FRQ,%.2f,AMP,%.2f,PHSE,%.2f'% (freq, ampli, fase)   ) 

gen.write('C1:OUTP ON') 

12

In [76]:
osci.write('DAT:ENC RPB')

osci.write('DAT:WID 1')         # data band width 1 byte

osci.write('HEADER OFF')

12

In [77]:
osci.write('SEL:CH1 ON')        # select:channel1 ON

osci.write('AUTOS EXEC')   # Equivalente a apretar el botón AUTOSET del equipo, 

time.sleep(5)      

osci.write('ACQ:STATE OFF')

15

In [83]:
osci.write('DAT:SOU CH1')       # data:source channel1

xze, xin, yze, ymu, yoff = osci.query_ascii_values('WFMOUTPRE:XZE?;XIN?;YZE?;YMU?;YOFF?;',separator=';') # waveform preamble information

time.sleep(2)           # breve pausa

time.sleep(2)           

dataCH1 = osci.query_binary_values('CURV?', datatype='B', container=np.array) 

t1 = xze + np.arange(len(dataCH1)) * xin             # Conversion a tiempo en segundos

VCH1 = (dataCH1 - yoff) * ymu + yze                 # Conversion a voltaje en Volts

data = {"tiempo":t, "voltaje":VCH1}

df = pd.DataFrame( data=data ) 

df.head() 


df.to_csv("./voltaje_CH13.csv", index=False )

In [84]:
osci.write('DAT:SOU CH2')       # data:source channel2

xze, xin, yze, ymu, yoff = osci.query_ascii_values('WFMOUTPRE:XZE?;XIN?;YZE?;YMU?;YOFF?;',separator=';') # waveform preamble information

time.sleep(2)           # breve pausa

time.sleep(2)           

dataCH2 = osci.query_binary_values('CURV?', datatype='B', container=np.array) 

t2 = xze + np.arange(len(dataCH2)) * xin             # Conversion a tiempo en segundos

VCH2 = (dataCH2 - yoff) * ymu + yze                 # Conversion a voltaje en Volts

data = {"tiempo":t, "voltaje":VCH1}

df = pd.DataFrame( data=data ) 

df.head()          

df.to_csv( "./voltaje_CH23.csv", index=False )